# Overview

This notebook is to develop the logic to connect to OneDrive (Sharepoint), get and push some files to it.

I am taking as reference the script `one_drive` from the [`data360-pipelines-python` repo](https://github.com/worldbank/data360-pipelines-python/blob/main/src/data360_pipelines_python/utils/one_drive.py)

# Setup

In [ ]:
import msal
from data360_ops.github import load_token
from pathlib import Path
import yaml
import requests

In [ ]:
# Import details for OneDrive connection
root_path = Path.cwd().parent.parent

In [ ]:
# Import system parameters and filter one drive params
with open(root_path / "systems_params.yml", "r") as f:
    params = yaml.safe_load(f)
    
onedrive_params = params['one_drive']
onedrive_params

# Development

## get_token_with_username

In [ ]:
def get_token_with_username(
		authority_url: str, 
		resource_url: str, 
        scope: str,
		client_id: str,
		tenant_id: str,
		username: str,
		password: str
	) -> dict:
    """Gets token after authentication to Azure Active Directory (ADD) using MSAL

    Args:
        authority_url (str): string with the url to authenticate
        resource_url (str): graph api url
        scope (str): scope
		client_id (str): client id of the application
		tenant_id (str): tenant id of the application
		username (str): username for authentication
		password (str): password for authentication
    Returns:
        Dictionary with authentication tokens and other relevant information
    """
    app = msal.PublicClientApplication(
        client_id=client_id, 
        authority=f"{authority_url}{tenant_id}")
    
    token = app.acquire_token_by_username_password(
        username=username, 
		password=password,        
        scopes=[f"{resource_url}{scope}"])

    return token

In [ ]:
token_path = root_path / "one_drive.token"
client_id = load_token(token_path, "CLIENT_ID")
tenant_id = load_token(token_path, "TENANT_ID")
username = load_token(token_path, "USERNAME")
password = load_token(token_path, "PASSWORD")

In [ ]:
# Get token
token = get_token_with_username(
	authority_url=onedrive_params['authority_url'],
	resource_url=onedrive_params['resource_url'],
	scope=onedrive_params['scope'],
	client_id=client_id,
	tenant_id=tenant_id,
	username=username,
	password=password
)

In [ ]:
# Get token
token = get_token_with_username(
	authority_url=onedrive_params['authority_url'],
	resource_url=onedrive_params['resource_url'],
	scope=onedrive_params['scope'],
	client_id="invalid_client_id",
	tenant_id="invalid_tenant_id",
	username="invalid_user",
	password="invalid_password"
)

## Download files

### Download a specific file from sites

In [ ]:
# Get token
token = get_token_with_username(
	authority_url=onedrive_params['authority_url'],
	resource_url=onedrive_params['resource_url'],
	scope=onedrive_params['scope'],
	client_id=client_id,
	tenant_id=tenant_id,
	username=username,
	password=password
)

access_token = token.get("access_token")

In [ ]:
# Define headers for authentication
headers = {"Authorization": f"Bearer {access_token}"}

In [ ]:
# Get ID of the drive
site = requests.get(
	f"{onedrive_params['resource_url']}{onedrive_params['api_version']}/sites/{onedrive_params['sharepoint_host_name']}:/sites/{onedrive_params['site_id']}:/drive",
	headers=headers
).json()

drive_id = site['id']

In [ ]:
# Download file
download_url = (
    f"{onedrive_params['resource_url']}{onedrive_params['api_version']}/"
    f"drives/{drive_id}/root:/{onedrive_params['template_mapping_file']}:/content"
)


response = requests.get(download_url, headers=headers)
response.raise_for_status()

with open("TemplateMappingFile.xlsx", "wb") as f:
    f.write(response.content)
    

### Download a specific file from personal space in OneDrive

In [ ]:
download_url = (
    f"{onedrive_params['resource_url']}{onedrive_params['api_version']}/"
    f"users/{onedrive_params['new_dataset_request_user_id']}/drive/root:/{onedrive_params['new_dataset_request_file']}:/content"
)


response = requests.get(download_url, headers=headers)
response.raise_for_status()

with open("NewDatasetResponses.xlsx", "wb") as f:
    f.write(response.content)
    

## List all files in a folder

### List files in public site

In [ ]:
# Get ID of the drive
site = requests.get(
	f"{onedrive_params['resource_url']}{onedrive_params['api_version']}/sites/{onedrive_params['sharepoint_host_name']}:/sites/{onedrive_params['site_id']}:/drive",
	headers=headers
).json()

drive_id = site['id']

In [ ]:
rel_path = onedrive_params['mapping_file_path']

In [ ]:
# rel_path = 'WB-Corporate/Data-Bank/Data360/DEC/Data Management'

In [ ]:
result = requests.get(
	f"{onedrive_params['resource_url']}{onedrive_params['api_version']}/drives/{drive_id}/root:/{rel_path}:/children", 
	headers=headers
)

dir_dict = {}
for x in result.json()['value']:
	dir_dict[x['name']] = x['id']
dir_dict

In [ ]:
if result.json()['value']:
	print("Files found:")

### List files in private drive

In [ ]:
url = (
    f"{onedrive_params['resource_url']}{onedrive_params['api_version']}/"
    f"users/{onedrive_params['test_user_id']}/drive/root:/{onedrive_params['test_folder_path']}:/children"
)

In [ ]:
result = requests.get(url, headers=headers)

dir_dict = {}
for x in result.json()['value']:
	dir_dict[x['name']] = x['id']
dir_dict

In [ ]:
url

In [ ]:
result

# Functions

## download_file

In [ ]:
def download_file(
		resource_url: str,
		api_version: str,
		site_user_id: str,
		file_path: str,
		output_filename: str,
		headers: dict,
		host_name: str = None,
		personal_drive: bool = False
	):
	"""This function downloads a file from OneDrive given the parameters.
	
	Args:
		resource_url (str): Graph API resource URL
		api_version (str): API version
		scope (str): Scope for authentication
		host_name (str): SharePoint host name
		site_user_id (str): User ID of the OneDrive site
		file_path (str): Path to the file in OneDrive
		output_filename (str): Local output filename
		headers (dict): Headers for authentication
		personal_drive (bool): Whether the file is in a personal drive or not. Default is False.
	Returns:
		None
	"""
	# Conditional to know where the file is stored
	# This is needed because the API endpoint is different
	if personal_drive:
		# This option is for personal drives

		# Download file
		download_url = (
			f"{resource_url}{api_version}/"
			f"users/{site_user_id}/drive/root:/{file_path}:/content"
		)
	else:
		# This options is for Team drive such as 'efiosfiles'

		# Throw error if host_name is not provided
		if host_name is None:
			raise ValueError("host_name must be provided for shared drives.")
		
		# Get ID of the drive
		site = requests.get(
			f"{resource_url}{api_version}/sites/{host_name}:/sites/{site_user_id}:/drive",
			headers=headers
		).json()

		drive_id = site['id']

		# Download file
		download_url = (
			f"{resource_url}{api_version}/"
			f"drives/{drive_id}/root:/{file_path}:/content"
		)
	# End if

	# Download the file
	response = requests.get(download_url, headers=headers)
	response.raise_for_status()

	# Export the file
	try:
		with open(f"{output_filename}", "wb") as f:
			f.write(response.content)
		success = True
		print(f"File downloaded successfully: {output_filename}")
	except Exception as e:
		print(f"Error downloading file: {e}")
		success = False
	
	return success


### Download file from shared drive

In [ ]:
# Import details for OneDrive connection
root_path = Path.cwd().parent.parent
token_path = root_path / "one_drive.token"
client_id = load_token(token_path, "CLIENT_ID")
tenant_id = load_token(token_path, "TENANT_ID")
username = load_token(token_path, "USERNAME")
password = load_token(token_path, "PASSWORD")

In [ ]:
## Import system parameters and filter one drive params
with open(root_path / "systems_params.yml", "r") as f:
    params = yaml.safe_load(f)
    
onedrive_params = params['one_drive']

In [ ]:
# Get token
token = get_token_with_username(
	authority_url=onedrive_params['authority_url'],
	resource_url=onedrive_params['resource_url'],
	scope=onedrive_params['scope'],
	client_id=client_id,
	tenant_id=tenant_id,
	username=username,
	password=password
)

access_token = token.get("access_token")

In [ ]:
# Define headers for authentication
headers = {"Authorization": f"Bearer {access_token}"}

In [ ]:
download_file(
	resource_url=onedrive_params['resource_url'],
	api_version=onedrive_params['api_version'],
	host_name=onedrive_params['sharepoint_host_name'],
	site_user_id=onedrive_params['site_id'],
	file_path=onedrive_params['template_mapping_file'],
	output_filename="TemplateMappingFile.xlsx",
	headers=headers,
	personal_drive=False
)

### Download file from personal drive

In [ ]:
download_file(
	resource_url=onedrive_params['resource_url'],
	api_version=onedrive_params['api_version'],
	site_user_id=onedrive_params['new_dataset_request_user_id'],
	file_path=onedrive_params['new_dataset_request_file'],
	output_filename="NewDatasetResponses.xlsx",
	headers=headers,
	personal_drive=True
)

## list_files

In [ ]:
def list_files(
        resource_url: str,
		api_version: str,
		site_user_id: str,
		folder_path: str,
		headers: dict,
		host_name: str = None, # type: ignore
		personal_drive: bool = False
	) -> dict:
	"""This function downloads a file from OneDrive given the parameters.
	
	Args:
		resource_url (str): Graph API resource URL
		api_version (str): API version
		site_user_id (str): User ID of the OneDrive site
		folder_path (str): Path to the folder in OneDrive
		headers (dict): Headers for authentication
		host_name (str): SharePoint host name
		personal_drive (bool): Whether the file is in a personal drive or not. Default is False.
	
	Raises:
		ValueError: If no files are found in the specified folder.
		
	Returns:
		Dict: Dictionary with file/folder names as keys and their IDs as values
	"""
	# Conditional to know where the file is stored
	# This is needed because the API endpoint is different
	if personal_drive:
		# This option is for personal drives

		# Define URL
		url = (
			f"{resource_url}{api_version}/"
			f"users/{site_user_id}/drive/root:/{folder_path}:/children"
		)
	else:
		# This options is for Team drive such as 'efiosfiles'

		# Throw error if host_name is not provided
		if host_name is None:
			raise ValueError("host_name must be provided for shared drives.")
		
		# Get ID of the drive
		site = requests.get(
			f"{resource_url}{api_version}/"
			f"sites/{host_name}:/sites/{site_user_id}:/drive",
			headers=headers
		).json()

		drive_id = site['id']

		# Download file
		url = (
			f"{resource_url}{api_version}/"
			f"drives/{drive_id}/root:/{folder_path}:/children"
		)
	# End if

	# Request list of files
	response = requests.get(url, headers=headers)
	response.raise_for_status()

	# Extract list of files and folders
	if response.json()['value']:
		dir_dict = {}
		for x in response.json()['value']:
			dir_dict[x['name']] = x['id']
		return dir_dict
	else: 
		raise ValueError("No files found in the specified folder.")


### List files from shared drive

In [ ]:
# Import details for OneDrive connection
root_path = Path.cwd().parent.parent
token_path = root_path / "one_drive.token"
client_id = load_token(token_path, "CLIENT_ID")
tenant_id = load_token(token_path, "TENANT_ID")
username = load_token(token_path, "USERNAME")
password = load_token(token_path, "PASSWORD")

In [ ]:
## Import system parameters and filter one drive params
with open(root_path / "systems_params.yml", "r") as f:
    params = yaml.safe_load(f)
    
onedrive_params = params['one_drive']

In [ ]:
# Get token
token = get_token_with_username(
	authority_url=onedrive_params['authority_url'],
	resource_url=onedrive_params['resource_url'],
	scope=onedrive_params['scope'],
	client_id=client_id,
	tenant_id=tenant_id,
	username=username,
	password=password
)

access_token = token.get("access_token")

In [ ]:
# Define headers for authentication
headers = {"Authorization": f"Bearer {access_token}"}

In [ ]:
files_dict = list_files(
	resource_url=onedrive_params['resource_url'],
	api_version=onedrive_params['api_version'],
	host_name=onedrive_params['sharepoint_host_name'],
	site_user_id=onedrive_params['site_id'],
	folder_path=onedrive_params['mapping_file_path'],
	headers=headers,
	personal_drive=False
)
files_dict

### List files from personal drive

In [ ]:
files_dict = list_files(
	resource_url=onedrive_params['resource_url'],
	api_version=onedrive_params['api_version'],
	site_user_id=onedrive_params['test_user_id'],
	folder_path=onedrive_params['test_folder_path'],
	headers=headers,
	personal_drive=True
)
files_dict